# Survey Validation — Public Review Draft

> **Review copy; not an executable public release.** The original notebook has 11 code cells. Ten original analysis/setup cells are preserved unchanged and one optional, three-household fixed-label sensitivity cell is replaced by a disclosure note. All saved notebook outputs are cleared. The notebook has **not** been rerun after curation.

The remaining code expects private Wave 1–3 survey data and locked clustering artifacts. It can **write household-level CSVs** (including `survey_outputs/socio_demographic_household_dataset_FINAL.csv`). Do **not** execute or commit generated exports to any public repository. See `REPRODUCIBILITY.md` in the repository root for the research and data-access boundaries.

In [ ]:
# ============================================================
# SURVEY ANALYSIS — FINAL SAFE SETUP
# Folder-aware version for:
#   wave_1/
#   wave_2/
#   wave_3/
# ============================================================

import os
import re
import glob
import warnings
import numpy as np
import pandas as pd

from scipy.stats import chi2_contingency, kruskal

warnings.filterwarnings("ignore")

BASE_DIR = "."
WAVE1_DIR = os.path.join(BASE_DIR, "wave_1")
WAVE2_DIR = os.path.join(BASE_DIR, "wave_2")
WAVE3_DIR = os.path.join(BASE_DIR, "wave_3")

OUTPUT_DIR = os.path.join(BASE_DIR, "survey_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOCKED_CLUSTER_PATH = "clean_fm_clusters.pkl"

CLUSTER_ORDER = [
    "Peak-Exposed",
    "Typical Household",
    "Heavy Continuous",
    "Daytime-Active"
]

CORE_FEATURES = [
    "evening_peak_ratio",
    "night_baseload_ratio",
    "load_uniformity",
    "load_factor",
    "leisure_usage_ratio",
    "log_daily_kwh"
]

def infer_archetype_name_map(feature_table, label_col="final_cluster"):
    """
    Infer stable archetype names from the locked feature profile.
    This avoids unsafe dependence on arbitrary KMeans numeric cluster IDs.
    """
    required = [
        label_col,
        "evening_peak_ratio",
        "night_baseload_ratio",
        "load_factor",
        "log_daily_kwh"
    ]

    missing = [c for c in required if c not in feature_table.columns]
    if missing:
        raise RuntimeError(
            "Cannot infer archetype names because required columns are missing: "
            + str(missing)
        )

    med = feature_table.groupby(label_col)[
        ["evening_peak_ratio", "night_baseload_ratio", "load_factor", "log_daily_kwh"]
    ].median()

    remaining = set(med.index.tolist())
    mapping = {}

    heavy_id = med["load_factor"].idxmax()
    mapping[heavy_id] = "Heavy Continuous"
    remaining.remove(heavy_id)

    peak_id = med.loc[list(remaining), "evening_peak_ratio"].idxmax()
    mapping[peak_id] = "Peak-Exposed"
    remaining.remove(peak_id)

    daytime_id = med.loc[list(remaining), "night_baseload_ratio"].idxmin()
    mapping[daytime_id] = "Daytime-Active"
    remaining.remove(daytime_id)

    typical_id = list(remaining)[0]
    mapping[typical_id] = "Typical Household"

    return dict(sorted(mapping.items(), key=lambda kv: kv[0]))

CLUSTER_NAMES = {}
CLUSTER_IDS = []
CLUSTER_NAME_TO_ID = {}

def clean_column_name(col):
    col = str(col).strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col

def list_csvs_in_folder(folder):
    if not os.path.exists(folder):
        print(f"Folder not found: {folder}")
        return []
    return sorted(glob.glob(os.path.join(folder, "*.csv")))

def to_numeric_safe(s):
    return pd.to_numeric(s, errors="coerce")

def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)

    order = np.argsort(pvals)
    ranked = pvals[order]

    adjusted = np.empty(n, dtype=float)
    cumulative_min = 1.0

    for i in range(n - 1, -1, -1):
        rank = i + 1
        value = ranked[i] * n / rank
        cumulative_min = min(cumulative_min, value)
        adjusted[order[i]] = cumulative_min

    return np.minimum(adjusted, 1.0)

def cramers_v_from_observed(observed):
    chi2, _, _, _ = chi2_contingency(observed, correction=False)
    n = observed.sum()
    denom = n * min(observed.shape[0] - 1, observed.shape[1] - 1)

    if denom <= 0:
        return np.nan

    return np.sqrt(chi2 / denom)

def epsilon_squared_kw(h_stat, n, k):
    if n <= k:
        return np.nan
    return max((h_stat - k + 1) / (n - k), 0)

def get_locked_labels():
    if "clean_fm" in globals():
        locked = clean_fm.copy()
    elif os.path.exists(LOCKED_CLUSTER_PATH):
        locked = pd.read_pickle(LOCKED_CLUSTER_PATH)
    else:
        raise RuntimeError("clean_fm not in memory and clean_fm_clusters.pkl not found.")

    locked = locked.copy()
    locked.index = locked.index.astype(str)

    if "final_cluster" not in locked.columns:
        raise RuntimeError("Locked clustering table must contain final_cluster.")

    labels = locked[["final_cluster"]].reset_index()

    if "index" in labels.columns:
        labels = labels.rename(columns={"index": "household_ID"})

    labels["household_ID"] = labels["household_ID"].astype(str)

    global CLUSTER_NAMES, CLUSTER_IDS, CLUSTER_NAME_TO_ID
    CLUSTER_NAMES = infer_archetype_name_map(locked, label_col="final_cluster")
    CLUSTER_NAME_TO_ID = {v: k for k, v in CLUSTER_NAMES.items()}
    CLUSTER_IDS = [CLUSTER_NAME_TO_ID[name] for name in CLUSTER_ORDER]

    labels["cluster_name"] = labels["final_cluster"].map(CLUSTER_NAMES)

    if labels["cluster_name"].isna().any():
        raise RuntimeError("Some final_cluster values could not be mapped to archetype names.")

    print("\nDynamic cluster-name map inferred from locked feature medians:")
    for cluster_id, cluster_name in CLUSTER_NAMES.items():
        print(f"  {cluster_id} -> {cluster_name}")

    return locked, labels

print("Setup complete.")
print("Wave 1 folder:", WAVE1_DIR)
print("Wave 2 folder:", WAVE2_DIR)
print("Wave 3 folder:", WAVE3_DIR)
print("Output folder:", OUTPUT_DIR)

In [ ]:
# ============================================================
# SURVEY FILE INVENTORY — WAVE FOLDERS ONLY
# Exploratory only; does not merge survey data
# ============================================================

print("=" * 80)
print("SURVEY FILE INVENTORY — WAVE FOLDERS ONLY")
print("=" * 80)

inventory_rows = []

for wave_name, folder in {
    "wave_1": WAVE1_DIR,
    "wave_2": WAVE2_DIR,
    "wave_3": WAVE3_DIR
}.items():

    files = list_csvs_in_folder(folder)

    for f in files:
        try:
            header = pd.read_csv(f, nrows=0)
            cols = list(header.columns)

            inventory_rows.append({
                "wave": wave_name,
                "file": os.path.basename(f),
                "n_columns": len(cols),
                "has_household_ID": "household_ID" in cols,
                "first_10_columns": ", ".join(map(str, cols[:10]))
            })

        except Exception as e:
            inventory_rows.append({
                "wave": wave_name,
                "file": os.path.basename(f),
                "n_columns": np.nan,
                "has_household_ID": False,
                "first_10_columns": f"ERROR: {e}"
            })

inventory_df = pd.DataFrame(inventory_rows)

print(inventory_df.to_string(index=False))

inventory_df.to_csv(
    os.path.join(OUTPUT_DIR, "survey_file_inventory_wave_folders_only.csv"),
    index=False
)

print("\nSaved:")
print("- survey_outputs/survey_file_inventory_wave_folders_only.csv")

print("\nSafe interpretation:")
print(
    "This inventory is restricted to wave_1, wave_2, and wave_3 folders. "
    "It avoids mixing survey files with smart-meter files or generated outputs."
)

In [ ]:
# ============================================================
# FINAL HOUSEHOLD-LEVEL SOCIO-DEMOGRAPHIC DATASET
# Uses Wave 1 only
# ============================================================

print("=" * 80)
print("FINAL HOUSEHOLD-LEVEL SOCIO-DEMOGRAPHIC DATASET")
print("=" * 80)

clean_fm, labels = get_locked_labels()

print("Locked clustered households:", labels["household_ID"].nunique())
print("\nLocked cluster distribution:")
print(
    labels["cluster_name"]
    .value_counts()
    .reindex(CLUSTER_ORDER)
    .fillna(0)
    .astype(int)
    .to_string()
)

# ---------- 1. Household-level numeric variables ----------
hh_path = os.path.join(WAVE1_DIR, "w1_household_information_and_history.csv")
demo_path = os.path.join(WAVE1_DIR, "w1_demographics.csv")
apps_path = os.path.join(WAVE1_DIR, "w1_appliances.csv")
ac_path = os.path.join(WAVE1_DIR, "w1_ac_roster.csv")

for path in [hh_path, demo_path, apps_path, ac_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required Wave 1 file not found: {path}")

hh = pd.read_csv(hh_path, low_memory=False)

required_hh = [
    "household_ID",
    "no_of_household_members",
    "floor_area",
    "total_monthly_expenditure_of_last_month"
]

missing = [c for c in required_hh if c not in hh.columns]
if missing:
    raise KeyError(f"Missing household columns: {missing}")

hh = hh[required_hh].drop_duplicates("household_ID").copy()
hh["household_ID"] = hh["household_ID"].astype(str)

for c in required_hh[1:]:
    hh[c] = pd.to_numeric(hh[c], errors="coerce")

hh["occupant_density_per_1000_sqft"] = np.where(
    hh["floor_area"] > 0,
    hh["no_of_household_members"] / hh["floor_area"] * 1000,
    np.nan
)

# ---------- 2. Member-level demographics aggregated to household ----------
demo = pd.read_csv(demo_path, low_memory=False)
demo["household_ID"] = demo["household_ID"].astype(str)

for c in ["age", "no_of_hours_stayed_at_home_during_last_week"]:
    if c in demo.columns:
        demo[c] = pd.to_numeric(demo[c], errors="coerce")

demo_aggs = {}

if "age" in demo.columns:
    demo_aggs["avg_member_age"] = ("age", "mean")

if "no_of_hours_stayed_at_home_during_last_week" in demo.columns:
    demo_aggs["avg_member_hours_at_home_week"] = (
        "no_of_hours_stayed_at_home_during_last_week",
        "mean"
    )

if demo_aggs:
    demo_hh = (
        demo.groupby("household_ID")
        .agg(**demo_aggs)
        .reset_index()
    )
else:
    demo_hh = demo[["household_ID"]].drop_duplicates()

# ---------- 3. Appliance ownership at household level ----------
apps = pd.read_csv(apps_path, low_memory=False)
apps["household_ID"] = apps["household_ID"].astype(str)

if "appliance_type" not in apps.columns:
    raise KeyError("w1_appliances.csv must contain appliance_type.")

APPLIANCES = {
    "Refrigerator": "owns_refrigerator",
    "TV": "owns_tv",
    "Rice cooker": "owns_rice_cooker",
    "Washing Machine": "owns_washing_machine",
    "Electric Iron including electric steam iron": "owns_iron",
    "Electric Kettle": "owns_electric_kettle",
    "Electric Water pump": "owns_water_pump",
    "Electric Oven": "owns_electric_oven",
    "Electric cook tops (induction cookers, Infra-red cookers, hot plates)": "owns_induction_cooker",
    "Laptops": "owns_laptop"
}

app_hh = apps[["household_ID"]].drop_duplicates().copy()

for appliance_name, output_col in APPLIANCES.items():
    owners = set(
        apps.loc[
            apps["appliance_type"].eq(appliance_name),
            "household_ID"
        ]
    )

    app_hh[output_col] = (
        app_hh["household_ID"].isin(owners).astype(int)
    )

app_count = (
    apps.groupby("household_ID")
    .size()
    .rename("appliance_roster_count")
    .reset_index()
)

app_hh = app_hh.merge(
    app_count,
    on="household_ID",
    how="left"
)

# ---------- 4. AC ownership ----------
ac = pd.read_csv(ac_path, low_memory=False)
ac["household_ID"] = ac["household_ID"].astype(str)

ac_owners = set(ac["household_ID"].dropna())

ac_hh = labels[["household_ID"]].copy()
ac_hh["owns_ac"] = (
    ac_hh["household_ID"].isin(ac_owners).astype(int)
)

# ---------- 5. Final one-row-per-household dataset ----------
socio_df = (
    labels[["household_ID", "final_cluster", "cluster_name"]]
    .merge(hh, on="household_ID", how="inner")
    .merge(demo_hh, on="household_ID", how="left")
    .merge(app_hh, on="household_ID", how="left")
    .merge(ac_hh, on="household_ID", how="left")
)

binary_cols = [
    c for c in socio_df.columns
    if c.startswith("owns_")
]

socio_df[binary_cols] = (
    socio_df[binary_cols]
    .fillna(0)
    .astype(int)
)

socio_df["appliance_roster_count"] = (
    socio_df["appliance_roster_count"]
    .fillna(0)
)

if socio_df["household_ID"].duplicated().any():
    raise ValueError("socio_df is not one row per household.")

print("\nFinal socio-demographic matched households:", len(socio_df))
print(
    socio_df["cluster_name"]
    .value_counts()
    .reindex(CLUSTER_ORDER)
    .fillna(0)
    .astype(int)
    .to_string()
)

socio_df.to_csv(
    os.path.join(OUTPUT_DIR, "socio_demographic_household_dataset_FINAL.csv"),
    index=False
)

print("\nSaved:")
print("- survey_outputs/socio_demographic_household_dataset_FINAL.csv")

print("\nSafe interpretation:")
print(
    "This is the final Wave 1 household-level survey validation dataset. "
    "It is one row per household and is matched to the locked smart-meter clusters."
)

print("\nOwnership-variable construction:")

print(
    "Appliance ownership indicators were derived from the complete "
    "Wave 1 household appliance roster. A household was coded as owning "
    "an appliance when the corresponding appliance type was recorded "
    "in its roster."
)

print(
    "AC ownership was inferred from household presence in the conditional "
    "Wave 1 AC equipment roster. Households without an AC roster record "
    "were coded as having no recorded AC ownership."
)

In [ ]:
# ============================================================
# SURVEY WAVE TIMING AUDIT
# Uses survey_dates.csv as timing/limitation evidence
# ============================================================

print("=" * 80)
print("SURVEY WAVE TIMING AUDIT")
print("=" * 80)

survey_date_candidates = [
    os.path.join(BASE_DIR, "survey_dates.csv"),
    os.path.join(WAVE1_DIR, "survey_dates.csv"),
    os.path.join(WAVE2_DIR, "survey_dates.csv"),
    os.path.join(WAVE3_DIR, "survey_dates.csv")
]

survey_date_file = None

for f in survey_date_candidates:
    if os.path.exists(f):
        survey_date_file = f
        break

if survey_date_file is None:
    print("No survey_dates.csv file found. Skipping timing audit.")

else:
    dates_df = pd.read_csv(survey_date_file, low_memory=False)

    if "household_ID" not in dates_df.columns:
        raise KeyError("survey_dates.csv must contain household_ID.")

    dates_df["household_ID"] = dates_df["household_ID"].astype(str)

    date_cols = [
        c for c in dates_df.columns
        if c.lower().startswith("wave")
    ]

    for c in date_cols:
        dates_df[c] = pd.to_datetime(
            dates_df[c],
            errors="coerce",
            dayfirst=False
        )

    locked_dates = labels[["household_ID", "cluster_name"]].merge(
        dates_df,
        on="household_ID",
        how="inner"
    )

    final_dates = socio_df[["household_ID", "cluster_name"]].merge(
        dates_df,
        on="household_ID",
        how="left"
    )

    golden_start = pd.Timestamp("2024-05-01")
    golden_end = pd.Timestamp("2024-12-01")  # exclusive upper bound

    rows = []

    for c in date_cols:
        all_dates = dates_df[c]
        locked_sample_dates = locked_dates[c]
        final_sample_dates = final_dates[c]

        rows.append({
            "wave": c,
            "all_households_with_date": all_dates.notna().sum(),
            "all_earliest": all_dates.min(),
            "all_latest": all_dates.max(),
            "all_inside_golden_window": (
                (all_dates >= golden_start) &
                (all_dates < golden_end)
            ).sum(),
            "locked_households_with_date": locked_sample_dates.notna().sum(),
            "final_survey_households_with_date": final_sample_dates.notna().sum(),
            "final_inside_golden_window": (
                (final_sample_dates >= golden_start) &
                (final_sample_dates < golden_end)
            ).sum(),
            "final_before_golden_window": (final_sample_dates < golden_start).sum(),
            "final_after_golden_window": (final_sample_dates >= golden_end).sum(),
            "final_median_date": final_sample_dates.median()
        })

    timing_summary = pd.DataFrame(rows)

    print("\nSurvey wave timing summary:")
    print(timing_summary.to_string(index=False))

    timing_summary.to_csv(
        os.path.join(OUTPUT_DIR, "survey_wave_timing_audit.csv"),
        index=False
    )

    print("\nSaved:")
    print("- survey_outputs/survey_wave_timing_audit.csv")

    print("\nSafe interpretation:")
    print(
        "Survey dates are used only as a timing audit. Wave 1 variables are used "
        "for final household-level validation, while Wave 2 and Wave 3 are treated "
        "as exploratory because their timing differs from the locked smart-meter window."
    )

In [ ]:
# ============================================================
# MATCHED VS UNMATCHED SURVEY COVERAGE AUDIT
# Formal test of differential survey availability by archetype
# ============================================================

print("=" * 80)
print("MATCHED VS UNMATCHED SURVEY COVERAGE AUDIT")
print("=" * 80)

locked_ids = set(
    labels["household_ID"]
    .astype(str)
)

matched_ids = set(
    socio_df["household_ID"]
    .astype(str)
)

unmatched_ids = locked_ids - matched_ids

print(
    "Locked clustered households:",
    len(locked_ids)
)

print(
    "Survey-matched households:",
    len(matched_ids)
)

print(
    "Unmatched clustered households:",
    len(unmatched_ids)
)

overall_match_rate = (
    len(matched_ids)
    / len(locked_ids)
    * 100
)

print(
    f"Survey match rate: "
    f"{overall_match_rate:.2f}%"
)


# ------------------------------------------------------------
# Household-level match indicator
# ------------------------------------------------------------

coverage_base = (
    labels[
        [
            "household_ID",
            "cluster_name"
        ]
    ]
    .copy()
)

coverage_base["household_ID"] = (
    coverage_base["household_ID"]
    .astype(str)
)

coverage_base[
    "survey_matched"
] = coverage_base[
    "household_ID"
].isin(matched_ids)


# ------------------------------------------------------------
# Descriptive coverage by archetype
# ------------------------------------------------------------

coverage_by_cluster = (
    coverage_base
    .groupby(
        "cluster_name"
    )[
        "survey_matched"
    ]
    .agg([
        "count",
        "sum",
        "mean"
    ])
    .reset_index()
)

coverage_by_cluster[
    "unmatched"
] = (
    coverage_by_cluster["count"]
    - coverage_by_cluster["sum"]
)

coverage_by_cluster[
    "match_rate_percent"
] = (
    coverage_by_cluster["mean"]
    * 100
)

coverage_by_cluster[
    "cluster_name"
] = pd.Categorical(
    coverage_by_cluster[
        "cluster_name"
    ],
    categories=CLUSTER_ORDER,
    ordered=True
)

coverage_by_cluster = (
    coverage_by_cluster
    .sort_values(
        "cluster_name"
    )
    .reset_index(
        drop=True
    )
)

print(
    "\nSurvey match coverage by locked archetype:"
)

print(
    coverage_by_cluster[
        [
            "cluster_name",
            "count",
            "sum",
            "unmatched",
            "match_rate_percent"
        ]
    ]
    .rename(
        columns={
            "count":
                "locked_households",
            "sum":
                "matched_households"
        }
    )
    .round(3)
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Formal chi-square test
# ------------------------------------------------------------

coverage_contingency = (
    pd.crosstab(
        coverage_base[
            "cluster_name"
        ],
        coverage_base[
            "survey_matched"
        ]
    )
    .reindex(
        index=CLUSTER_ORDER,
        columns=[
            False,
            True
        ],
        fill_value=0
    )
)

coverage_contingency.columns = [
    "unmatched",
    "matched"
]

(
    chi2_coverage,
    p_coverage,
    dof_coverage,
    expected_coverage
) = chi2_contingency(
    coverage_contingency,
    correction=False
)

coverage_cramers_v = (
    cramers_v_from_observed(
        coverage_contingency
        .to_numpy()
    )
)

minimum_expected_count = float(
    expected_coverage.min()
)

coverage_test_summary = pd.DataFrame([
    {
        "chi_square":
            chi2_coverage,
        "degrees_of_freedom":
            dof_coverage,
        "p_value":
            p_coverage,
        "cramers_v":
            coverage_cramers_v,
        "minimum_expected_count":
            minimum_expected_count,
        "overall_match_rate_percent":
            overall_match_rate,
        "locked_households":
            len(locked_ids),
        "matched_households":
            len(matched_ids)
    }
])

print(
    "\nMatched/unmatched contingency table:"
)

print(
    coverage_contingency
    .to_string()
)

print(
    "\nFormal survey-coverage test:"
)

print(
    coverage_test_summary
    .round(6)
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

coverage_by_cluster.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "survey_match_coverage_by_cluster.csv"
    ),
    index=False
)

coverage_contingency.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "survey_match_contingency_table.csv"
    )
)

coverage_test_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "survey_match_bias_test.csv"
    ),
    index=False
)

print("\nSaved:")

print(
    "- survey_outputs/"
    "survey_match_coverage_by_cluster.csv"
)

print(
    "- survey_outputs/"
    "survey_match_contingency_table.csv"
)

print(
    "- survey_outputs/"
    "survey_match_bias_test.csv"
)


# ------------------------------------------------------------
# Conservative interpretation
# ------------------------------------------------------------

print("\nInterpretation:")

if p_coverage < 0.05:

    print(
        "Survey availability differs significantly across "
        "the locked archetypes."
    )

    print(
        f"Cramer's V = {coverage_cramers_v:.3f}, "
        "indicating meaningful differential coverage."
    )

    print(
        "The Wave 1 matched sample is therefore not fully "
        "representative of the complete clustered population."
    )

    print(
        "Subsequent survey findings must be interpreted as "
        "matched-sample external contextual validation."
    )

else:

    print(
        "The test did not detect statistically significant "
        "differences in survey availability across archetypes."
    )


In [ ]:
# ============================================================
# FINAL SURVEY VARIABLE MISSINGNESS AUDIT
# ============================================================

print("=" * 80)
print("FINAL SURVEY VARIABLE MISSINGNESS AUDIT")
print("=" * 80)

survey_variable_cols = [
    "no_of_household_members",
    "floor_area",
    "occupant_density_per_1000_sqft",
    "total_monthly_expenditure_of_last_month",
    "avg_member_age",
    "avg_member_hours_at_home_week",
    "appliance_roster_count"
] + binary_cols

survey_variable_cols = [
    c for c in survey_variable_cols
    if c in socio_df.columns
]

missingness_rows = []

for c in survey_variable_cols:
    missingness_rows.append({
        "variable": c,
        "non_missing_n": socio_df[c].notna().sum(),
        "missing_n": socio_df[c].isna().sum(),
        "missing_percent": socio_df[c].isna().mean() * 100
    })

missingness_df = pd.DataFrame(missingness_rows)

print(missingness_df.round(2).to_string(index=False))

missingness_df.to_csv(
    os.path.join(OUTPUT_DIR, "final_survey_variable_missingness_audit.csv"),
    index=False
)

print("\nSaved:")
print("- survey_outputs/final_survey_variable_missingness_audit.csv")

print("\nSafe interpretation:")
print(
    "Association tests use available non-missing records for each variable. "
    "Therefore, sample size can differ across variables."
)

In [ ]:
# ============================================================
# EXPLORATORY DESCRIPTIVE PROFILING BY LOCKED CLUSTER
# Descriptive only; no causal claims
# ============================================================

print("=" * 80)
print("EXPLORATORY DESCRIPTIVE PROFILING BY LOCKED CLUSTER")
print("=" * 80)

continuous_cols = [
    "no_of_household_members",
    "floor_area",
    "occupant_density_per_1000_sqft",
    "total_monthly_expenditure_of_last_month",
    "avg_member_age",
    "avg_member_hours_at_home_week",
    "appliance_roster_count"
]

continuous_cols = [
    c for c in continuous_cols
    if c in socio_df.columns and socio_df[c].notna().sum() > 0
]

binary_cols = [
    c for c in socio_df.columns
    if c.startswith("owns_")
]

continuous_summary = []

for col in continuous_cols:
    tmp = (
        socio_df
        .groupby("cluster_name")[col]
        .agg(["count", "mean", "median", "std"])
        .reset_index()
    )
    tmp.insert(0, "variable", col)
    continuous_summary.append(tmp)

continuous_summary = (
    pd.concat(continuous_summary, ignore_index=True)
    if continuous_summary else pd.DataFrame()
)

binary_summary = []

for col in binary_cols:
    tmp = (
        socio_df
        .groupby("cluster_name")[col]
        .agg(["count", "mean"])
        .reset_index()
        .rename(columns={"mean": "share"})
    )
    tmp["share_percent"] = tmp["share"] * 100
    tmp.insert(0, "variable", col)
    binary_summary.append(tmp)

binary_summary = (
    pd.concat(binary_summary, ignore_index=True)
    if binary_summary else pd.DataFrame()
)

print("\nContinuous descriptive summary:")
if not continuous_summary.empty:
    print(continuous_summary.round(3).to_string(index=False))

print("\nBinary descriptive summary:")
if not binary_summary.empty:
    print(binary_summary.round(3).to_string(index=False))

continuous_summary.to_csv(
    os.path.join(OUTPUT_DIR, "exploratory_continuous_profile_by_cluster.csv"),
    index=False
)

binary_summary.to_csv(
    os.path.join(OUTPUT_DIR, "exploratory_binary_profile_by_cluster.csv"),
    index=False
)

print("\nSaved:")
print("- survey_outputs/exploratory_continuous_profile_by_cluster.csv")
print("- survey_outputs/exploratory_binary_profile_by_cluster.csv")

print("\nSafe interpretation:")
print(
    "These are descriptive cluster profiles only. They should not be interpreted "
    "as causal explanations of electricity-use archetypes."
)

In [ ]:
# ============================================================
# FINAL ROBUST SOCIO-DEMOGRAPHIC TESTS
# Kruskal-Wallis + epsilon squared
# Permutation Chi-square + Cramer's V
# Benjamini-Hochberg FDR correction
# ============================================================

print("=" * 80)
print("FINAL ROBUST SOCIO-DEMOGRAPHIC TESTS")
print("=" * 80)

RANDOM_STATE = 42
N_PERMUTATIONS = 50000
rng = np.random.default_rng(RANDOM_STATE)

continuous_cols = [
    "no_of_household_members",
    "floor_area",
    "occupant_density_per_1000_sqft",
    "total_monthly_expenditure_of_last_month",
    "avg_member_age",
    "avg_member_hours_at_home_week",
    "appliance_roster_count"
]

binary_cols = [
    c for c in socio_df.columns
    if c.startswith("owns_")
]

results = []

# ---------- Continuous variables ----------
for col in [c for c in continuous_cols if c in socio_df.columns]:

    d = (
        socio_df[["final_cluster", col]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    cluster_ids = sorted(d["final_cluster"].unique())

    groups = [
        d.loc[d["final_cluster"] == cluster_id, col].values
        for cluster_id in cluster_ids
    ]

    if len(groups) < 2:
        continue

    h_stat, p_value = kruskal(*groups)

    n = len(d)
    k = len(groups)

    epsilon_squared = epsilon_squared_kw(
        h_stat,
        n=n,
        k=k
    )

    results.append({
        "variable": col,
        "test": "Kruskal-Wallis",
        "statistic": h_stat,
        "p_raw": p_value,
        "effect_size": epsilon_squared,
        "effect_size_name": "epsilon_squared",
        "minimum_expected_count": np.nan,
        "n": n,
        "asymptotic_p_reference": np.nan
    })

# ---------- Binary categorical variables ----------
for col in binary_cols:

    d = (
        socio_df[["final_cluster", col]]
        .dropna()
        .copy()
    )

    d[col] = pd.to_numeric(d[col], errors="coerce")
    d = d[d[col].isin([0, 1])]

    if d.empty:
        continue

    cluster_codes = d["final_cluster"].astype(int).to_numpy()
    binary_values = d[col].astype(int).to_numpy()

    cluster_levels = np.sort(np.unique(cluster_codes))

    if len(cluster_levels) < 2 or len(np.unique(binary_values)) < 2:
        continue

    recode = {
        cluster_id: position
        for position, cluster_id in enumerate(cluster_levels)
    }

    group_codes = np.array(
        [recode[x] for x in cluster_codes],
        dtype=int
    )

    group_sizes = np.bincount(
        group_codes,
        minlength=len(cluster_levels)
    )

    positive_counts = np.bincount(
        group_codes,
        weights=binary_values,
        minlength=len(cluster_levels)
    )

    observed = np.column_stack([
        group_sizes - positive_counts,
        positive_counts
    ])

    try:
        chi2_observed, asymptotic_p, dof, expected = chi2_contingency(
            observed,
            correction=False
        )
    except Exception:
        continue

    n = observed.sum()

    cramers_v = cramers_v_from_observed(observed)

    permuted_statistics = np.empty(
        N_PERMUTATIONS,
        dtype=float
    )

    for permutation_index in range(N_PERMUTATIONS):

        shuffled = rng.permutation(binary_values)

        perm_positive = np.bincount(
            group_codes,
            weights=shuffled,
            minlength=len(cluster_levels)
        )

        perm_table = np.column_stack([
            group_sizes - perm_positive,
            perm_positive
        ])

        try:
            permuted_statistics[permutation_index] = chi2_contingency(
                perm_table,
                correction=False
            )[0]
        except Exception:
            permuted_statistics[permutation_index] = np.nan

    permuted_statistics = permuted_statistics[
        np.isfinite(permuted_statistics)
    ]

    if len(permuted_statistics) == 0:
        permutation_p = np.nan
    else:
        permutation_p = (
            1 + np.sum(permuted_statistics >= chi2_observed)
        ) / (
            len(permuted_statistics) + 1
        )

    results.append({
        "variable": col,
        "test": "Permutation Chi-square",
        "statistic": chi2_observed,
        "p_raw": permutation_p,
        "effect_size": cramers_v,
        "effect_size_name": "Cramers_V",
        "minimum_expected_count": expected.min(),
        "n": int(n),
        "asymptotic_p_reference": asymptotic_p
    })

robust_tests = pd.DataFrame(results)

if robust_tests.empty:
    raise RuntimeError("No robust survey tests could be run.")

robust_tests["p_fdr_bh"] = bh_fdr(
    robust_tests["p_raw"].fillna(1.0).to_numpy()
)

robust_tests["significant_after_fdr"] = (
    robust_tests["p_fdr_bh"] < 0.05
)

robust_tests = robust_tests.sort_values(
    ["p_fdr_bh", "variable"]
).reset_index(drop=True)

print("\nRobust final survey tests:")
print(robust_tests.round(6).to_string(index=False))

print("\nFDR-significant variables:")
sig_vars = robust_tests.loc[
    robust_tests["significant_after_fdr"],
    "variable"
].tolist()

if sig_vars:
    for v in sig_vars:
        print("-", v)
else:
    print("None")

robust_tests.to_csv(
    os.path.join(OUTPUT_DIR, "survey_final_tests_ROBUST_FDR_corrected.csv"),
    index=False
)

print("\nSaved:")
print("- survey_outputs/survey_final_tests_ROBUST_FDR_corrected.csv")


print(
    f"\nPermutation iterations per binary variable: "
    f"{N_PERMUTATIONS:,}"
)

print(
    "Minimum attainable Monte Carlo p-value:",
    round(
        1 / (
            N_PERMUTATIONS + 1
        ),
        7
    )
)

print("\nSafe interpretation:")
print(
    "These tests assess association between Wave 1 household-level survey "
    "variables and locked smart-meter clusters. Results are external validation "
    "evidence only and should not be interpreted causally. Effect sizes should "
    "be reported alongside FDR-corrected p-values."
)

In [ ]:
# ============================================================
# WAVE 2 / WAVE 3 HOUSEHOLD-UNIT AUDIT
#
# Later-wave files are not used for final inference.
# Files containing multiple rows per household are not
# summarized by archetype because doing so would weight
# households according to their number of roster records.
# ============================================================

print("=" * 80)
print("WAVE 2 / WAVE 3 HOUSEHOLD-UNIT AUDIT")
print("=" * 80)


def audit_later_wave_files(
    folder,
    wave_name
):

    files = list_csvs_in_folder(
        folder
    )

    audit_rows = []

    print(
        "\n" + "-" * 80
    )

    print(
        f"{wave_name.upper()} FILE AUDIT"
    )

    print(
        "-" * 80
    )

    for file_path in files:

        file_name = os.path.basename(
            file_path
        )

        try:

            df = pd.read_csv(
                file_path,
                low_memory=False
            )

            if "household_ID" not in df.columns:

                audit_rows.append({
                    "wave":
                        wave_name,
                    "file":
                        file_name,
                    "rows":
                        len(df),
                    "unique_households":
                        np.nan,
                    "households_with_multiple_rows":
                        np.nan,
                    "maximum_rows_per_household":
                        np.nan,
                    "matched_locked_households":
                        np.nan,
                    "summary_status":
                        "Skipped: no household_ID"
                })

                print(
                    f"{file_name}: skipped because "
                    "household_ID is absent."
                )

                continue

            df[
                "household_ID"
            ] = (
                df["household_ID"]
                .astype(str)
            )

            rows_per_household = (
                df.groupby(
                    "household_ID"
                )
                .size()
            )

            unique_households = int(
                df["household_ID"]
                .nunique()
            )

            multiple_row_households = int(
                (
                    rows_per_household > 1
                ).sum()
            )

            maximum_rows = int(
                rows_per_household.max()
            )

            matched_households = int(
                df.loc[
                    df["household_ID"]
                    .isin(
                        labels[
                            "household_ID"
                        ]
                    ),
                    "household_ID"
                ]
                .nunique()
            )

            if multiple_row_households > 0:

                status = (
                    "Not summarized by archetype: "
                    "multiple rows per household"
                )

            else:

                status = (
                    "One row per household; eligible "
                    "for separate descriptive analysis"
                )

            audit_rows.append({
                "wave":
                    wave_name,
                "file":
                    file_name,
                "rows":
                    len(df),
                "unique_households":
                    unique_households,
                "households_with_multiple_rows":
                    multiple_row_households,
                "maximum_rows_per_household":
                    maximum_rows,
                "matched_locked_households":
                    matched_households,
                "summary_status":
                    status
            })

            print(
                f"{file_name} | "
                f"rows={len(df):,} | "
                f"unique households="
                f"{unique_households:,} | "
                f"households with multiple rows="
                f"{multiple_row_households:,} | "
                f"status={status}"
            )

        except Exception as error:

            audit_rows.append({
                "wave":
                    wave_name,
                "file":
                    file_name,
                "rows":
                    np.nan,
                "unique_households":
                    np.nan,
                "households_with_multiple_rows":
                    np.nan,
                "maximum_rows_per_household":
                    np.nan,
                "matched_locked_households":
                    np.nan,
                "summary_status":
                    f"Error: {error}"
            })

            print(
                f"{file_name}: {error}"
            )

    return pd.DataFrame(
        audit_rows
    )


wave2_unit_audit = audit_later_wave_files(
    WAVE2_DIR,
    "wave_2"
)

wave3_unit_audit = audit_later_wave_files(
    WAVE3_DIR,
    "wave_3"
)

later_wave_unit_audit = pd.concat(
    [
        wave2_unit_audit,
        wave3_unit_audit
    ],
    ignore_index=True
)

print(
    "\nCombined later-wave household-unit audit:"
)

print(
    later_wave_unit_audit
    .to_string(
        index=False
    )
)

later_wave_unit_audit.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "wave2_wave3_household_unit_audit.csv"
    ),
    index=False
)

print("\nSaved:")

print(
    "- survey_outputs/"
    "wave2_wave3_household_unit_audit.csv"
)

print("\nInterpretation:")

print(
    "Wave 2 and Wave 3 files are retained only as "
    "supplementary data-structure evidence."
)

print(
    "Files with multiple rows per household are not "
    "summarized by archetype because row-level summaries "
    "would give greater weight to households with more "
    "members, appliances, rooms or roster records."
)

print(
    "The final socio-demographic validation remains based "
    "on the one-row-per-household Wave 1 dataset."
)


In [ ]:
# ============================================================
# EXPLORATORY SOLAR / EXPORT SENSITIVITY CHECK
# Descriptive only; does not alter final locked clustering
# ============================================================

print("=" * 80)
print("EXPLORATORY SOLAR / EXPORT SENSITIVITY CHECK")
print("=" * 80)

gen_path = os.path.join(
    WAVE1_DIR,
    "w1_electricity_generation_water_heating_cooking.csv"
)

if os.path.exists(gen_path):
    gen_df = pd.read_csv(gen_path, low_memory=False)
    gen_df["household_ID"] = gen_df["household_ID"].astype(str)

    solar_col = "generate_electicity_using_solar_energy"

    if solar_col in gen_df.columns:
        solar_merged = labels[["household_ID", "final_cluster", "cluster_name"]].merge(
            gen_df[["household_ID", solar_col]],
            on="household_ID",
            how="inner"
        )

        print("\nSelf-reported solar ownership in matched clustering population:")
        print("Total matched:", len(solar_merged))
        print(solar_merged[solar_col].value_counts(dropna=False).to_string())

        print("\nSolar ownership by cluster:")
        ct = pd.crosstab(
            solar_merged["cluster_name"],
            solar_merged[solar_col],
            margins=True
        )
        print(ct.to_string())

        pct = (
            pd.crosstab(
                solar_merged["cluster_name"],
                solar_merged[solar_col],
                normalize="index"
            ) * 100
        )

        print("\nPercentage by cluster:")
        print(pct.round(1).to_string())

        pct.to_csv(
            os.path.join(OUTPUT_DIR, "exploratory_self_reported_solar_by_cluster.csv")
        )

        print("\nSaved:")
        print("- survey_outputs/exploratory_self_reported_solar_by_cluster.csv")
    else:
        print(f"{solar_col} not found in {gen_path}")
else:
    print("Wave 1 electricity generation file not found. Skipping self-reported solar check.")

if "solar_export_delta" in globals():
    measured_export_flag = (
        solar_export_delta.loc[solar_export_delta > 1.0]
        .index
        .astype(str)
    )

    measured_export_df = pd.DataFrame({
        "household_ID": measured_export_flag,
        "measured_export_flag": 1
    })

    export_cluster_check = labels[["household_ID", "cluster_name"]].merge(
        measured_export_df,
        on="household_ID",
        how="left"
    )

    export_cluster_check["measured_export_flag"] = (
        export_cluster_check["measured_export_flag"]
        .fillna(0)
        .astype(int)
    )

    export_table = (
        export_cluster_check
        .groupby("cluster_name")["measured_export_flag"]
        .agg(["count", "sum", "mean"])
        .reset_index()
    )

    export_table["mean_percent"] = export_table["mean"] * 100

    print("\nMeasured export-register flag by locked cluster:")
    print(export_table.round(3).to_string(index=False))

    export_table.to_csv(
        os.path.join(OUTPUT_DIR, "exploratory_measured_export_flag_by_cluster.csv"),
        index=False
    )

    print("\nSaved:")
    print("- survey_outputs/exploratory_measured_export_flag_by_cluster.csv")
else:
    print("\nsolar_export_delta not found in memory. Skipping measured export-register flag check.")

print("\nSafe interpretation:")
print(
    "Self-reported solar ownership and measured export-register movement are "
    "related but non-equivalent indicators. This exploratory check should not "
    "be used to claim that all solar households were identified or that the "
    "two indicators must match exactly."
)

## Private case-specific sensitivity investigation — omitted

The original final code cell performs an optional fixed-label descriptive sensitivity check by excluding three specified self-reported solar households. It retains the previously locked cluster assignments and **does not retrain K-Means**. The code and its household identifiers are withheld from this public review draft; the original cell is preserved in the private research copy.

The exclusion experiment and its outputs cannot be reproduced from this public notebook alone. No results are asserted or recalculated here.